In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import gc
import sys
import os
sys.path.append('../')
from AIID_notebooks.inflapy.immunoFunc import *

In [2]:
print(sc.__version__)

1.9.1


In [3]:
path="/lustre/scratch117/cellgen/cellgeni/TIC-starsolo/tic-1808/"

In [4]:
s_path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/"

##### GSE121380

In [5]:
#GSE121380/GSM3433578/output/Velocyto/filtered/

pheno = pd.read_csv(s_path+"GSE121380/pheno.csv")

In [6]:
pheno

,Unnamed: 0,title,geo_accession,source_name_ch1,characteristics_ch1.1,patient diagnosis:ch1
0,GSM3433578,HB1_GSM3433578,GSM3433578,colon,cell markers: CD45+CD3+CD19-,paediatric onset inflammatory bowel disease (P...
1,GSM3433579,HB2_GSM3433579,GSM3433579,colon,cell markers: CD45+CD3-CD19+,paediatric onset inflammatory bowel disease (P...
2,GSM3433580,HB3-1_GSM3433580,GSM3433580,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...
3,GSM3433581,HB3-2_GSM3433581,GSM3433581,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...
4,GSM3433582,HB4_GSM3433582,GSM3433582,colon,cell markers: CD45-,paediatric onset inflammatory bowel disease (P...
5,GSM3433583,S279_GSM3433583,GSM3433583,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...
6,GSM3433584,S280_GSM3433584,GSM3433584,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...


In [7]:
#'donor_id', 'tissue', 'dataset_id'GSE121380_Huang_2019

pheno.rename(columns={"title":"donor_id", "source_name_ch1": "tissue"}, inplace=True)

In [8]:
pheno["dataset_id"] = np.repeat("GSE121380_Huang_2019", pheno.shape[0])
pheno["cell_source"] = np.repeat("Colon sorted CD45+", pheno.shape[0])
pheno["disease"] = np.repeat("Paediatric IBD", pheno.shape[0])

In [9]:
pheno

,Unnamed: 0,donor_id,geo_accession,tissue,characteristics_ch1.1,patient diagnosis:ch1,dataset_id,cell_source,disease
0,GSM3433578,HB1_GSM3433578,GSM3433578,colon,cell markers: CD45+CD3+CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
1,GSM3433579,HB2_GSM3433579,GSM3433579,colon,cell markers: CD45+CD3-CD19+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
2,GSM3433580,HB3-1_GSM3433580,GSM3433580,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
3,GSM3433581,HB3-2_GSM3433581,GSM3433581,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
4,GSM3433582,HB4_GSM3433582,GSM3433582,colon,cell markers: CD45-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
5,GSM3433583,S279_GSM3433583,GSM3433583,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
6,GSM3433584,S280_GSM3433584,GSM3433584,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD


##### Immune compartment

In [10]:
#only immune compartments
pheno_immune = pheno[pheno['characteristics_ch1.1']!='cell markers: CD45-']
pheno_immune

,Unnamed: 0,donor_id,geo_accession,tissue,characteristics_ch1.1,patient diagnosis:ch1,dataset_id,cell_source,disease
0,GSM3433578,HB1_GSM3433578,GSM3433578,colon,cell markers: CD45+CD3+CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
1,GSM3433579,HB2_GSM3433579,GSM3433579,colon,cell markers: CD45+CD3-CD19+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
2,GSM3433580,HB3-1_GSM3433580,GSM3433580,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
3,GSM3433581,HB3-2_GSM3433581,GSM3433581,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
5,GSM3433583,S279_GSM3433583,GSM3433583,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
6,GSM3433584,S280_GSM3433584,GSM3433584,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD


In [11]:
pheno_immune.reset_index(inplace=True)

In [12]:
dataset="GSE121380/"

In [13]:
pheno_immune

,index,Unnamed: 0,donor_id,geo_accession,tissue,characteristics_ch1.1,patient diagnosis:ch1,dataset_id,cell_source,disease
0,0,GSM3433578,HB1_GSM3433578,GSM3433578,colon,cell markers: CD45+CD3+CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
1,1,GSM3433579,HB2_GSM3433579,GSM3433579,colon,cell markers: CD45+CD3-CD19+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
2,2,GSM3433580,HB3-1_GSM3433580,GSM3433580,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
3,3,GSM3433581,HB3-2_GSM3433581,GSM3433581,colon,cell markers: CD45+CD3-CD19-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
4,5,GSM3433583,S279_GSM3433583,GSM3433583,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD
5,6,GSM3433584,S280_GSM3433584,GSM3433584,colon,cell markers: CD45+,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45+,Paediatric IBD


In [14]:
for i in range(pheno_immune.shape[0]):
    samp = pheno_immune.geo_accession[i]
    print("Processing "+samp)
    adata = sc.read_mtx(path+dataset+samp+"/output/Gene/filtered/matrix.mtx.gz")

    var_names = np.genfromtxt(path+dataset+samp+"/output/Gene/filtered/genes.tsv.gz",dtype=str)
    var_names


    var_names[:,0]

        #print('observation names...')
    obs_names = np.genfromtxt(path+dataset+samp+"/output/Gene/filtered/barcodes.tsv.gz",dtype=str)
    obs_names

    obs_names.shape

    adata = adata.transpose()

    adata.var_names = var_names[:,1]
    adata.obs_names = obs_names

    adata.var_names_make_unique()

    adata.var[['ensembl', 'gene_symbol']] = var_names[:,0:2]
    
    adata.obs["dataset_id"] = pheno_immune.dataset_id[i]
    adata.obs["donor_id"] = pheno_immune.donor_id[i]
    adata.obs["tissue"] = pheno_immune.tissue[i]
    adata.obs["cell_source"] = pheno_immune.cell_source[i]
    adata.obs["disease"] = pheno_immune.disease[i]
    
    if 'adata_conc' in locals():
        adata.var_names_make_unique()
        adata_conc.var_names_make_unique()
        adata_conc = sc.concat([adata,adata_conc], index_unique="-")
        
    else:
        adata_conc = adata

    print(adata_conc.shape)

Processing GSM3433578
(13654, 36601)
Processing GSM3433579
(24454, 36601)
Processing GSM3433580
(34139, 36601)
Processing GSM3433581
(39938, 36601)
Processing GSM3433583
(44623, 36601)
Processing GSM3433584
(50287, 36601)


In [15]:
adata_conc.write_h5ad(s_path+"Immune_compartment_sorted_from_Tissue/GSE121380_Huang_2019_immune.h5ad")

##### Non immune

In [16]:
del adata_conc
del adata
gc.collect()

1042

In [17]:
#only immune compartments
pheno_nonimmune = pheno[pheno['characteristics_ch1.1']=='cell markers: CD45-']
pheno_nonimmune.reset_index(inplace=True)


In [18]:
pheno_nonimmune["cell_source"] = np.repeat("Colon sorted CD45 neg", pheno_nonimmune.shape[0])

<ipython-input-18-d9a34eaf5c20>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pheno_nonimmune["cell_source"] = np.repeat("Colon sorted CD45 neg", pheno_nonimmune.shape[0])


In [19]:
pheno_nonimmune

,index,Unnamed: 0,donor_id,geo_accession,tissue,characteristics_ch1.1,patient diagnosis:ch1,dataset_id,cell_source,disease
0,4,GSM3433582,HB4_GSM3433582,GSM3433582,colon,cell markers: CD45-,paediatric onset inflammatory bowel disease (P...,GSE121380_Huang_2019,Colon sorted CD45 neg,Paediatric IBD


In [20]:
for i in range(pheno_nonimmune.shape[0]):
    samp = pheno_nonimmune.geo_accession[i]
    print("Processing "+samp)
    adata = sc.read_mtx(path+dataset+samp+"/output/Gene/filtered/matrix.mtx.gz")

    var_names = np.genfromtxt(path+dataset+samp+"/output/Gene/filtered/genes.tsv.gz",dtype=str)
    var_names


    var_names[:,0]

        #print('observation names...')
    obs_names = np.genfromtxt(path+dataset+samp+"/output/Gene/filtered/barcodes.tsv.gz",dtype=str)
    obs_names

    obs_names.shape

    adata = adata.transpose()

    adata.var_names = var_names[:,1]
    adata.obs_names = obs_names

    adata.var_names_make_unique()

    adata.var[['ensembl', 'gene_symbol']] = var_names[:,0:2]
    
    adata.obs["dataset_id"] = pheno_nonimmune.dataset_id[i]
    adata.obs["donor_id"] = pheno_nonimmune.donor_id[i]
    adata.obs["tissue"] = pheno_nonimmune.tissue[i]
    adata.obs["cell_source"] = pheno_nonimmune.cell_source[i]
    adata.obs["disease"] = pheno_nonimmune.disease[i]
    
    if 'adata_conc' in locals():
        adata.var_names_make_unique()
        adata_conc.var_names_make_unique()
        adata_conc = sc.concat([adata,adata_conc], index_unique="-")
        
    else:
        adata_conc = adata

    print(adata_conc.shape)

Processing GSM3433582
(2178, 36601)


In [21]:
adata_conc.write_h5ad(s_path+"Tissue_or_Nonimmune/GSE121380_Huang_2019_Non_immune.h5ad")